# Baseline Models: Logistic Regression and Decision Tree
Owner: Anupama Singh

Data: ECRI dataset (`data/ECRIDataSet.csv`). Download from https://github.com/ecri-data/ECRI-Dataset/tree/main/ECRI

## 1. Load and explore

In [3]:
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("../data/ECRIDataSet.csv")
print(df.shape)
df.isna().mean().sort_values(ascending=False).head(10)

(6803, 30)


RMwdaRS_target    0.860944
RMwidRS_target    0.860944
RMwdaRS           0.860944
RMwidRS           0.860944
Age1b             0.860944
ORFwc             0.788035
NWFwrc            0.565486
NWFcls            0.565486
grp_rate          0.419962
rcmistot          0.402323
dtype: float64

## 2. Keep only labeled students
Only students with a target value can be used for training (about 946).

We leave out `grp_rate`, `rcmistot`, `gnrl_fid`, `Tier2MinRd` because they are measured during the school year (data leakage risk).

In [4]:
df = df[df["RMwidRS_target"].notna()]
print("Labeled students:", len(df))

features = ["Gender", "Tier", "LEP", "SpEd", "Tx", "Age1b", "Tier2_N",
            "TKPctCrct", "TYearsTeaching", "NWFcls", "NWFwrc", "ORFwc",
            "SAwrS", "SAsrS", "SAtoS", "RMwidRS", "RMwdaRS"]
X = df[features]
groups = df["SchlID"]          # split by school so test schools are unseen
cv = GroupKFold(n_splits=5)

targets = {"Word Identification": "RMwidRS_target", "Word Attack": "RMwdaRS_target"}
for name, col in targets.items():
    print(name, df[col].value_counts(normalize=True).round(2).to_dict())

Labeled students: 946
Word Identification {1.0: 0.53, 0.0: 0.47}
Word Attack {1.0: 0.58, 0.0: 0.42}


## 3. Define the models

In [5]:
def logistic_regression():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000))

def decision_tree(depth):
    return make_pipeline(SimpleImputer(strategy="median"),
                         DecisionTreeClassifier(max_depth=depth, min_samples_leaf=20, random_state=0))

## 4. Tune Decision Tree depth

In [6]:
for name, col in targets.items():
    y = df[col]
    for depth in range(2, 9):
        auc = cross_val_score(decision_tree(depth), X, y, cv=cv, groups=groups, scoring="roc_auc").mean()
        print(f"{name:20s} depth={depth}  AUC={auc:.3f}")

Word Identification  depth=2  AUC=0.601
Word Identification  depth=3  AUC=0.606
Word Identification  depth=4  AUC=0.605
Word Identification  depth=5  AUC=0.629
Word Identification  depth=6  AUC=0.627
Word Identification  depth=7  AUC=0.627
Word Identification  depth=8  AUC=0.619
Word Attack          depth=2  AUC=0.527
Word Attack          depth=3  AUC=0.536
Word Attack          depth=4  AUC=0.551
Word Attack          depth=5  AUC=0.555
Word Attack          depth=6  AUC=0.556
Word Attack          depth=7  AUC=0.555
Word Attack          depth=8  AUC=0.556


## 5. Final baseline results

In [7]:
rows = []
for name, col in targets.items():
    y = df[col]
    best_depth = max(range(2, 9), key=lambda d: cross_val_score(
        decision_tree(d), X, y, cv=cv, groups=groups, scoring="roc_auc").mean())
    for model_name, model in [("Logistic Regression", logistic_regression()),
                              (f"Decision Tree (depth={best_depth})", decision_tree(best_depth))]:
        auc = cross_val_score(model, X, y, cv=cv, groups=groups, scoring="roc_auc")
        acc = cross_val_score(model, X, y, cv=cv, groups=groups, scoring="accuracy")
        rows.append({"target": name, "model": model_name,
                     "AUC": round(auc.mean(), 3), "AUC_std": round(auc.std(), 3),
                     "Accuracy": round(acc.mean(), 3)})

results = pd.DataFrame(rows)
results.to_csv("../results/baselines.csv", index=False)
results

,target,model,AUC,AUC_std,Accuracy
0,Word Identification,Logistic Regression,0.659,0.030,0.616
1,Word Identification,Decision Tree (depth=5),0.629,0.020,0.612
2,Word Attack,Logistic Regression,0.618,0.038,0.603
3,Word Attack,Decision Tree (depth=6),0.556,0.033,0.573
